In [1]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [7]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd

In [18]:
# fetch dataset 
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 
  
# data (as pandas dataframes) 
X = diabetes_130_us_hospitals_for_years_1999_2008.data.features 
y = diabetes_130_us_hospitals_for_years_1999_2008.data.targets 
  
# metadata 
print(diabetes_130_us_hospitals_for_years_1999_2008.metadata) 
  
# variable information 
print(diabetes_130_us_hospitals_for_years_1999_2008.variables) 

{'uci_id': 296, 'name': 'Diabetes 130-US Hospitals for Years 1999-2008', 'repository_url': 'https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008', 'data_url': 'https://archive.ics.uci.edu/static/public/296/data.csv', 'abstract': 'The dataset represents ten years (1999-2008) of clinical care at 130 US hospitals and integrated delivery networks. Each row concerns hospital records of patients diagnosed with diabetes, who underwent laboratory, medications, and stayed up to 14 days. The goal is to determine the early readmission of the patient within 30 days of discharge.\nThe problem is important for the following reasons. Despite high-quality evidence showing improved clinical outcomes for diabetic patients who receive various preventive and therapeutic interventions, many patients do not receive them. This can be partially attributed to arbitrary diabetes management in hospital environments, which fail to attend to glycemic control. Failure to provide pro

c:\Users\test\AppData\Local\Programs\Python\Python313\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [ ]:
ids = diabetes_130_us_hospitals_for_years_1999_2008.data.ids
temp = ids.join(X)
df = temp.join(y)

In [ ]:
# save dataframe with original data
df.to_csv('original_data.csv')

In [22]:
print(df.columns)

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')


In [23]:
print(df.groupby('weight').size())

weight
>200            3
[0-25)         48
[100-125)     625
[125-150)     145
[150-175)      35
[175-200)      11
[25-50)        97
[50-75)       897
[75-100)     1336
dtype: int64


In [24]:
# most common admitting medical specialties
df['medical_specialty'].value_counts(dropna=False)

medical_specialty
NaN                       49949
InternalMedicine          14635
Emergency/Trauma           7565
Family/GeneralPractice     7440
Cardiology                 5352
                          ...  
Dermatology                   1
SportsMedicine                1
Speech                        1
Perinatology                  1
Neurophysiology               1
Name: count, Length: 73, dtype: int64

In [21]:
# find number of missing values for each column with missing values + overall percentage missing

# number of rows
n_rows = df.shape[0]
# column names 
columns = df.columns
col_percent_missing = []

# calculate percentage of missing values for each column 
for col in columns:
    col_na = df[col].isna().sum()
    col_percent = (col_na/n_rows)*100
    col_percent_missing.append((col, col_percent))

col_to_drop = []

# print columns with missing values
for col in col_percent_missing:
    if (col[1] > 0):
        print(f'{col[0]}: {col[1]:.2f}% missing')
        if (col[1] > 75):
            col_to_drop.append(col[0])


race: 2.23% missing
weight: 96.86% missing
payer_code: 39.56% missing
medical_specialty: 49.08% missing
diag_1: 0.02% missing
diag_2: 0.35% missing
diag_3: 1.40% missing
max_glu_serum: 94.75% missing
A1Cresult: 83.28% missing


In [25]:
# drop columns with > 75% missing values
df_new = df.drop(columns=col_to_drop)

In [29]:
# drop columns that are not needed for model prediction
drop_col = ['payer_code', 'num_lab_procedures']
df_temp = df_new.drop(columns=drop_col)

In [30]:
# remove rows with missing race and diagnosis information
df_updated = df_temp.dropna(subset = ['race', 'diag_1', 'diag_2', 'diag_3'])

In [31]:
# check for duplicate rows 
ids['encounter_id'].duplicated().any()

np.False_

In [32]:
# check for inconsistencies in data formatting (inconsistent capitalization, spacing, etc)
for col in df_updated.columns:
    print(df_updated[col].value_counts(dropna=False))

encounter_id
149190       1
64410        1
500364       1
16680        1
35754        1
            ..
443847548    1
443847782    1
443854148    1
443857166    1
443867222    1
Name: count, Length: 98053, dtype: int64
patient_nbr
88785891     39
1660293      23
23199021     23
88227540     23
23643405     22
             ..
183087545     1
188574944     1
140199494     1
120975314     1
175429310     1
Name: count, Length: 68630, dtype: int64
race
Caucasian          75079
AfricanAmerican    18881
Hispanic            1984
Other               1484
Asian                625
Name: count, dtype: int64
gender
Female             52833
Male               45219
Unknown/Invalid        1
Name: count, dtype: int64
age
[70-80)     25306
[60-70)     21809
[80-90)     16702
[50-60)     16697
[40-50)      9265
[30-40)      3548
[90-100)     2717
[20-30)      1478
[10-20)       466
[0-10)         65
Name: count, dtype: int64
admission_type_id
1    52178
3    18194
2    17543
6     5135
5     4661
8    

In [33]:
# remove one row with unknown gender value
df_updated.drop(df_updated[df_updated['gender'] == 'Unknown/Invalid'].index, inplace=True)

In [34]:
# remove null/unknown values from numerically mapped columns 

# admission - remove 5, 6, 8
df_updated = df_updated[~df_updated['admission_type_id'].isin([5, 6, 8])]

# discharge - remove 18, 25, 26
df_updated = df_updated[~df_updated['discharge_disposition_id'].isin([18, 25, 26])]

# admission source - remove 9, 15, 17, 20, 21  
df_updated = df_updated[~df_updated['admission_source_id'].isin([9, 15, 17, 20, 21])]

In [35]:
# change readmitted to binary values, where < 30 readmission is 1, otherwise 0

import numpy as np
df_updated['readmitted'] = np.where(df_updated['readmitted'] == '<30', 1, 0) 

In [36]:
df_updated['readmitted'].value_counts(dropna=False)

readmitted
0    73857
1     9397
Name: count, dtype: int64

In [37]:
# create new features 

# create col for total hospitalizations 
df_updated['num_hospitalizations'] = df_updated.groupby('patient_nbr')['patient_nbr'].transform('count')

# procedures/length of stay
df_updated['avg_procedure'] = df_updated['num_procedures'] / df_updated['time_in_hospital'] 

# number of total visits
df_updated['total_visits'] = df_updated['number_outpatient'] + df_updated['number_emergency'] + df_updated['number_inpatient']

# number of med changes 
df_updated['num_med_changes'] = df_updated.loc[:, 'metformin':'metformin-pioglitazone'].isin(['Down', 'Up']).sum(axis=1)

# number of med increases 
df_updated['num_med_increase'] = df_updated.loc[:, 'metformin':'metformin-pioglitazone'].isin(['Up']).sum(axis=1)

In [ ]:
# drop medical speciality and medication info not needed for model prediction

df_final = df_updated.drop(columns=df_updated.loc[:, 'metformin':'metformin-pioglitazone'].columns)
df_final = df_final.drop(columns=['medical_specialty'])
df_final.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'change', 'diabetesMed',
       'readmitted', 'num_hospitalizations', 'avg_procedure', 'total_visits',
       'num_med_changes', 'num_med_increase'],
      dtype='str')

In [55]:
df_final.head(10)

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_procedures,...,diag_3,number_diagnoses,change,diabetesMed,readmitted,num_hospitalizations,avg_procedure,total_visits,num_med_changes,num_med_increase
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,0,...,255,9,Ch,Yes,0,1,0.000000,0,1,1
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,5,...,V27,6,No,Yes,0,1,2.500000,3,0,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,1,...,403,7,Ch,Yes,0,1,0.500000,0,1,1
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,0,...,250,5,Ch,Yes,0,1,0.000000,0,0,0
5,35754,82637451,Caucasian,Male,[50-60),2,1,2,3,6,...,250,9,No,Yes,0,1,2.000000,0,0,0
6,55842,84259809,Caucasian,Male,[60-70),3,1,2,4,1,...,V45,7,Ch,Yes,0,1,0.250000,0,0,0
7,63768,114882984,Caucasian,Male,[70-80),1,1,7,5,0,...,250,8,No,Yes,0,1,0.000000,0,0,0
8,12522,48330783,Caucasian,Female,[80-90),2,1,4,13,2,...,38,8,Ch,Yes,0,1,0.153846,0,0,0
9,15738,63555939,Caucasian,Female,[90-100),3,3,4,12,3,...,486,8,Ch,Yes,0,1,0.250000,0,0,0
10,28236,89869032,AfricanAmerican,Female,[40-50),1,1,7,9,2,...,996,9,No,Yes,0,1,0.222222,0,0,0


In [56]:
# save processed dataframe 
df_final.to_csv('final_data.csv')

In [38]:
icd_data = pd.read_json('icd_codes.json')

In [39]:
icd_data

,code,desc,children
0,00-00,"Procedures And Interventions , Not Elsewhere C...","[{'code': '00', 'desc': 'Procedures And Interv..."
1,01-05,Operations On The Nervous System,"[{'code': '01', 'desc': 'Incision And Excision..."
2,06-07,Operations On The Endocrine System,"[{'code': '06', 'desc': 'Operations On Thyroid..."
3,08-16,Operations On The Eye,"[{'code': '08', 'desc': 'Operations On Eyelids..."
4,17-17,Other Miscellaneous Diagnostic And Therapeutic...,"[{'code': '17', 'desc': 'Other Miscellaneous P..."
5,18-20,Operations On The Ear,"[{'code': '18', 'desc': 'Operations On Externa..."
6,21-29,"Operations On The Nose, Mouth, And Pharynx","[{'code': '21', 'desc': 'Operation On Nose', '..."
7,30-34,Operations On The Respiratory System,"[{'code': '30', 'desc': 'Excision Of Larynx', ..."
8,35-39,Operations On The Cardiovascular System,"[{'code': '35', 'desc': 'Operations On Valves ..."
9,40-41,Operations On The Hemic And Lymphatic System,"[{'code': '40', 'desc': 'Operations On Lymphat..."


In [ ]:
# load in icd code data
import json
with open('icd_codes.json') as file:
    icd_raw_data = json.load(file)

In [ ]:
# test line to see how data is organized 
record = icd_raw_data[0]
print(record.get('children')[0].get('children')[0])

{'code': '00.0', 'desc': 'Therapeutic Ultrasound', 'children': [{'code': '00.01', 'desc': 'Therapeutic ultrasound of vessels of head and neck', 'children': []}, {'code': '00.02', 'desc': 'Therapeutic ultrasound of heart', 'children': []}, {'code': '00.03', 'desc': 'Therapeutic ultrasound of peripheral vascular vessels', 'children': []}, {'code': '00.09', 'desc': 'Other therapeutic ultrasound', 'children': []}]}


In [42]:
# extract icd data from nested format

icd_data = []

for category in icd_raw_data:
    primary_category = category.get('desc')
    sub_categories = category.get('children')
    for sub_category in sub_categories:
        sub_category_desc = sub_category.get('desc')
        codes = sub_category.get('children')
        for value in codes:
            code = value.get('code')
            code_desc = value.get('desc')
            icd_data.append({'code': code, 'description': code_desc, 'primary_category': primary_category, 'subcategory': sub_category_desc})

In [43]:
# save icd data to dataframe 
icd_code_df = pd.DataFrame(icd_data)

In [44]:
# export icd_dataframe 
icd_code_df.to_csv('icd_code.csv')